In [4]:
import os
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from google.colab import files
uploaded = files.upload()

Saving clause_dataset.csv to clause_dataset.csv


In [5]:
# CONFIG
MODEL_NAME = "roberta-base"


MODEL_OUTPUT = (
    "/content/models/clause_classifier"
)

MAX_LENGTH = 512

BATCH_SIZE = 8

EPOCHS = 2

LEARNING_RATE = 2e-5


In [6]:
# LOAD DATA
print("=" * 60)
print("LOADING PROCESSED DATA")
print("=" * 60)

df = pd.read_csv("clause_dataset.csv")

print("Dataset shape:",df.shape)

LOADING PROCESSED DATA
Dataset shape: (18579, 3)


In [7]:
# TRAIN / VALIDATE SPLIT
train_df, val_df, = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_id"])
print("Traning samples:", len(train_df))
print("Validation samples:", len(val_df))

Traning samples: 14863
Validation samples: 3716


In [8]:
# HUGGINGFACE DATASET
train_dataset = Dataset.from_pandas(train_df[["text","label_id"]],
                                    preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["text","label_id"]],
                                     preserve_index=False)
train_dataset = train_dataset.rename_column("label_id", "labels")
val_dataset = val_dataset.rename_column("label_id", "labels")

In [9]:
# TOKENIZER

print("\nLoading RoBERTa tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True
)


Loading RoBERTa tokenizer...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/14863 [00:00<?, ? examples/s]

Map:   0%|          | 0/3716 [00:00<?, ? examples/s]

In [10]:
# MODEL

number_of_labels = df["label_id"].nunique()

print(
    "\nNumber of clause categories:",
    number_of_labels
)

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=number_of_labels
)


Number of clause categories: 38


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# TRAINING CONFIG

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=False
)

In [12]:
# TRAINER

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    processing_class=tokenizer
)


In [13]:
# TRAIN

print("\n" + "=" * 60)
print("STARTING ROBERTA FINE-TUNING")
print("=" * 60)

trainer.train()


STARTING ROBERTA FINE-TUNING


Epoch,Training Loss,Validation Loss
1,2.491523,2.541453
2,2.459118,2.472353


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3716, training_loss=2.6233862311257483, metrics={'train_runtime': 3089.7613, 'train_samples_per_second': 9.621, 'train_steps_per_second': 1.203, 'total_flos': 7823767286673408.0, 'train_loss': 2.6233862311257483, 'epoch': 2.0})